# 01 · Data Exploration: 3W Dataset
**Goal (Phase 1):** understand what each event *looks like* in the sensors, which sensors exist on which wells, and how much data we have.

Before running: `python scripts/01_build_inventory.py`

In [ ]:
import sys; sys.path.insert(0, "../src")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from wellsentinel.config import load_config
from wellsentinel.data import EVENT_NAMES, VARIABLES, VARIABLE_DESCRIPTIONS, load_instance

cfg = load_config()
FIG = cfg["paths"]["figures_dir"]
inv = pd.read_csv(cfg["paths"]["interim_dir"] / "inventory.csv")
inv.head()

## 1. How many instances of each event, by source?

In [ ]:
tab = pd.crosstab(inv["event"], inv["instance_type"]).loc[[EVENT_NAMES[i] for i in range(10)]]
ax = tab.plot.barh(stacked=True, figsize=(9, 5)); ax.invert_yaxis()
ax.set_xlabel("instances"); ax.set_title("3W v2.0.0 instances by event and source")
plt.tight_layout(); plt.savefig(FIG / "instances_by_event.png", dpi=150); tab

## 2. Hours of real data per well

In [ ]:
real = inv[inv.instance_type == "real"]
(real.groupby("well")["n_rows"].sum() / 3600).sort_values(ascending=False).head(15).plot.bar(figsize=(10, 3), title="Hours of real data per well (top 15)")

## 3. What does each event look like? (one real instance per event)
Change `SENSORS` to explore. Shaded = transient phase, darker = steady state.

In [ ]:
SENSORS = ["P-PDG", "P-TPT", "T-TPT", "P-MON-CKP"]

def plot_instance(path, sensors=SENSORS, title=""):
    df = load_instance(path)
    cls = df["class"].astype("float")
    fig, axes = plt.subplots(len(sensors), 1, figsize=(11, 2.0 * len(sensors)), sharex=True)
    for ax, s in zip(axes, sensors):
        ax.plot(df.index, df[s], lw=0.8); ax.set_ylabel(s, fontsize=8)
        ax.fill_between(df.index, *ax.get_ylim(), where=cls >= 100, color="orange", alpha=0.15, step="mid")
        ax.fill_between(df.index, *ax.get_ylim(), where=(cls > 0) & (cls < 100), color="red", alpha=0.12, step="mid")
    axes[0].set_title(title); plt.tight_layout(); return fig

for label in range(1, 10):
    cand = real[real.label == label]
    if len(cand):
        r = cand.iloc[0]
        fig = plot_instance(r.path, title=f"{label} – {EVENT_NAMES[label]}  ({r.instance_id})")
        fig.savefig(FIG / f"event_{label}_example.png", dpi=120); plt.show()

## 4. Sensor availability: which sensors are present on which wells?

In [ ]:
rows = []
for well, g in real.groupby("well"):
    df = load_instance(g.iloc[0].path)
    rows.append(df[VARIABLES].notna().mean().rename(well))
avail = pd.DataFrame(rows)
plt.figure(figsize=(12, 8)); plt.imshow(avail.values, aspect="auto", cmap="Greens")
plt.yticks(range(len(avail)), avail.index, fontsize=7); plt.xticks(range(len(VARIABLES)), VARIABLES, rotation=90, fontsize=7)
plt.title("Share of non-missing values (first instance of each well)"); plt.colorbar(); plt.tight_layout()
plt.savefig(FIG / "sensor_availability.png", dpi=150)

## 5. Notes: write your physics interpretation here, then copy it into `docs/DATA_CARD.md`
- Event 8 (hydrate, production line): ...
- Event 3 (severe slugging): ...